# Lab Activity 1: Linear Systems with NumPy
**Course:** CSE473: Computational Intelligence — Mechatronics Engineering and Automation Program  
**Prepares you for:** Lab Assignment 01 — Solving Linear Systems with NumPy

## 🎯 Learning objectives
By the end of this lab you will be able to:
- Build a random linear system $Ax = b$ with NumPy, reproducibly, using `np.random.default_rng`
- Solve **square** systems ($m = n$) exactly with `np.linalg.solve`
- Solve **overdetermined** systems ($m > n$) with least squares via `np.linalg.lstsq`
- Measure solution quality with the residual $\|Ax - b\|$ and recognize ill-conditioned matrices
- Chain these steps into an experiment runner for many sizes ($m \geq n$) — the assignment's core loop

⏱ **Estimated time: ~45 minutes**

## How this lab works
- The lab is split into **Parts**; each Part teaches one topic.
- Each Part starts with a short explanation plus a runnable **✏️ Worked example** — run it, tweak it, break it.
- Then you solve **🎯 Problems**. Read the task, write your code in the starter cell, and try it before opening any hints.
- Stuck? Open the **💡 Hint** blocks below each problem — Hint 1 is a nudge, Hint 2 names the approach. They get more specific as you go.
- Verify yourself with the **🧪 Self-check** cells — they run deterministic checks and fail with guiding messages until your solution is right.
- Truly stuck? The **✅ Reveal solution** block at the bottom of each hint section shows full working code.

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)
print("Setup OK — NumPy is ready.")

## Part 1: NumPy arrays & building a linear system (≈10 min)

A linear system $Ax = b$ packs $m$ equations in $n$ unknowns into one matrix equation: row $i$ of $A$ holds the coefficients of equation $i$, and $b_i$ is its right-hand side.

- $A$ has shape $(m, n)$; $x$ has shape $(n,)$; $b$ has shape $(m,)$.
- Picture it as: $m$ rows → $m$ equations → $m$ entries in $b$; $n$ columns → $n$ unknowns.
- The assignment uses **random** systems: coefficients $a \sim U(-1, 1)$ and right-hand sides $b \sim U(-1, 3)$ (uniform draws).
- `rng = np.random.default_rng(seed)` gives reproducible draws; `rng.uniform(low, high, size=...)` draws from a uniform range.
- `@` is matrix-vector multiplication: once you have a candidate $x$, `A @ x` should land back on $b$.

In [ ]:
# --- Worked example: build a tiny system by hand, then randomly ---
# A 2x2 system:  2*x1 + x2 = 5 ;  x1 + 3*x2 = 10
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])      # one ROW per equation, one COLUMN per unknown
b = np.array([5.0, 10.0])       # right-hand side: one entry per equation

print("A shape:", A.shape, "| b shape:", b.shape)
print("A @ (ones) =", A @ np.ones(2))   # '@' = matrix multiplication

# The same shapes, filled with reproducible random numbers:
rng = np.random.default_rng(42)          # seed -> identical draw on every run
A_rand = rng.uniform(-1, 1, size=(3, 3)) # a's ~ U(-1, 1)
b_rand = rng.uniform(-1, 3, size=3)      # b's ~ U(-1, 3)
print("\nRandom A (3x3):\n", A_rand)
print("Random b (3,):", b_rand)
print("All A entries in [-1, 1]?", bool(A_rand.min() >= -1 and A_rand.max() <= 1))

## 🎯 Problem 1.1 — Build a random linear system

**Given:** integers `m` (number of equations) and `n` (number of unknowns), plus an optional `seed`.

**Required:** write `generate_linear_system(m, n, seed=None)` returning a tuple `(A, b)` where

- `A` is a NumPy array of shape `(m, n)` with entries drawn uniformly from $[-1, 1]$,
- `b` is a NumPy array of shape `(m,)` with entries drawn uniformly from $[-1, 3]$,
- the same `seed` always produces the same system.

**Expected output:** `A.shape == (m, n)`, `b.shape == (m,)`, all values inside their ranges. This function powers every later problem in this lab.

In [ ]:
def generate_linear_system(m, n, seed=None):
    """Generate a random linear system Ax = b.

    Entries of A ~ U(-1, 1); entries of b ~ U(-1, 3).

    Args:
        m: number of equations (rows of A)
        n: number of unknowns (columns of A)
        seed: optional seed for reproducibility

    Returns:
        (A, b): A has shape (m, n), b has shape (m,)
    """
    # TODO: Your code here
    pass

# Demo call (produces output once you implement the function)
result = generate_linear_system(3, 3, seed=0)
if result is not None:
    A_demo, b_demo = result
    print("A_demo:\n", A_demo)
    print("b_demo:", b_demo)
else:
    print("Implement generate_linear_system to power this demo.")

<details>
<summary>💡 Hint 1 — where to start</summary>

You need one random-number generator and two draws from it. Which NumPy object gives *reproducible* draws when you hand it a seed? The worked example in Part 1 shows it.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

1. Create `rng = np.random.default_rng(seed)`.
2. `A = rng.uniform(-1, 1, size=(m, n))` — **size** carries both dimensions.
3. `b = rng.uniform(-1, 3, size=m)` — a 1-D draw takes one dimension.
4. `return A, b`.
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def generate_linear_system(m, n, seed=None):
    """Generate a random linear system Ax = b (a ~ U(-1,1), b ~ U(-1,3))."""
    rng = np.random.default_rng(seed)
    A = rng.uniform(-1, 1, size=(m, n))
    b = rng.uniform(-1, 3, size=m)
    return A, b
```
</details>

In [ ]:
# 🧪 Self-check for Problem 1.1
res = generate_linear_system(4, 3, seed=42)
assert res is not None, "❌ generate_linear_system returned None — did you replace the 'pass'? See Hint 2 in Part 1."
A1, b1 = res
assert A1.shape == (4, 3), f"❌ A has shape {A1.shape}, expected (4, 3). Check the size argument of rng.uniform."
assert b1.shape == (4,), f"❌ b has shape {b1.shape}, expected (4,)."
assert A1.min() >= -1.0 and A1.max() <= 1.0, "❌ A entries must lie in [-1, 1]."
assert b1.min() >= -1.0 and b1.max() <= 3.0, "❌ b entries must lie in [-1, 3]."

A1b, b1b = generate_linear_system(4, 3, seed=42)
assert np.allclose(A1, A1b) and np.allclose(b1, b1b), "❌ Same seed must give the same system — did you pass seed into default_rng?"
A2, b2 = generate_linear_system(4, 3, seed=43)
assert not np.allclose(A1, A2), "❌ Different seeds should give different systems."

print("✅ Problem 1.1 passed — you can build reproducible random systems.")

## Part 2: Solving square systems (m = n) (≈10 min)

When $A$ is **square** ($m = n$) and nonsingular, exactly one $x$ satisfies $Ax = b$ — geometrically, the intersection point of $n$ hyperplanes.

- `np.linalg.solve(A, b)` finds it via LU factorization: fast and exact up to floating-point round-off.
- Do **not** compute `np.linalg.inv(A) @ b` — slower and less accurate; `solve` is the professional choice.
- Always verify by substitution: `A @ x` should equal `b` to about $10^{-15}$.
- If $A$ is singular (linearly dependent rows), `solve` raises `np.linalg.LinAlgError` instead of returning garbage.

In [ ]:
# --- Worked example: solve the 2x2 system from Part 1 exactly ---
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([5.0, 10.0])

x = np.linalg.solve(A, b)              # exact solution of the square system
print("x =", x)                        # expect [1. 3.]

residual = np.linalg.norm(A @ x - b)   # substitution check
print("||A x - b|| =", residual)       # ~1e-16: floating-point round-off only

# A singular matrix raises instead of returning nonsense:
try:
    np.linalg.solve(np.array([[1.0, 2.0], [2.0, 4.0]]), b)
except np.linalg.LinAlgError as e:
    print("Singular input raises:", e)

## 🎯 Problem 2.1 — Solve a square system

**Given:** a square matrix `A` of shape `(n, n)` and a right-hand side `b` of shape `(n,)`.

**Required:** write `solve_square(A, b)` returning the exact solution `x` of $Ax = b$ as a 1-D array of shape `(n,)`.

**Expected output:** `A @ x` matches `b` to roughly $10^{-15}$ when substituted back.

In [ ]:
def solve_square(A, b):
    """Solve the square system Ax = b exactly.

    Args:
        A: square coefficient matrix, shape (n, n)
        b: right-hand side, shape (n,)

    Returns:
        x: solution vector, shape (n,)
    """
    # TODO: Your code here
    pass

# Demo call
demo_sys = generate_linear_system(3, 3, seed=1)
if demo_sys is not None:
    A_demo, b_demo = demo_sys
    x_demo = solve_square(A_demo, b_demo)
    if x_demo is not None:
        print("x_demo:", x_demo)
        print("check ||A x - b|| =", np.linalg.norm(A_demo @ x_demo - b_demo))
    else:
        print("Implement solve_square to see the solution.")
else:
    print("Finish Problem 1.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — which tool</summary>

One NumPy function solves a square system directly — you saw it in this Part's worked example. You do NOT need to invert the matrix.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

The function is `np.linalg.solve(A, b)`: it takes the matrix and the right-hand side, and returns the solution vector. One line: `return np.linalg.solve(A, b)`.
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def solve_square(A, b):
    """Solve the square system Ax = b exactly with np.linalg.solve."""
    return np.linalg.solve(A, b)
```
</details>

In [ ]:
# 🧪 Self-check for Problem 2.1
res = generate_linear_system(3, 3, seed=5)
assert res is not None, "❌ generate_linear_system returned None — finish Problem 1.1 first."
A_sq, b_sq = res
x_sq = solve_square(A_sq, b_sq)
assert x_sq is not None, "❌ solve_square returned None — replace the 'pass'. See Hint 2 in Part 2."
x_sq = np.asarray(x_sq, dtype=float)
assert x_sq.shape == (3,), f"❌ x has shape {x_sq.shape}, expected (3,)."
assert np.linalg.norm(A_sq @ x_sq - b_sq) < 1e-8, "❌ A @ x does not match b — did you use np.linalg.solve (not lstsq) for square systems? See Hint 2 in Part 2."
print("✅ Problem 2.1 passed — square systems solve exactly.")

## Part 3: Overdetermined systems & least squares (m > n) (≈10 min)

With **more equations than unknowns** ($m > n$), the equations generally contradict each other — picture several lines in the plane that cannot all pass through one point. No $x$ satisfies every row exactly.

- The **least-squares** solution minimizes the residual $\|Ax - b\|$: geometrically it projects $b$ onto the column space of $A$.
- `np.linalg.lstsq(A, b, rcond=None)` returns a tuple `(x, residuals, rank, sv)` — the solution is the **first** element.
- For a genuinely inconsistent system the minimum residual is **not** near zero — it is just the smallest achievable distance. (How to *prove* optimality: the residual is orthogonal to every column of $A$, i.e. $A^T(Ax-b) \approx 0$.)
- Rule of thumb: $m = n$ → `solve`; $m > n$ → `lstsq`.

In [ ]:
# --- Worked example: 5 equations, 2 unknowns -> no exact solution exists ---
rng = np.random.default_rng(0)
A = rng.uniform(-1, 1, size=(5, 2))
b = rng.uniform(-1, 3, size=5)

x, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)
print("x =", x)
print("rank:", rank, "| singular values:", sv)
print("||A x - b|| =", np.linalg.norm(A @ x - b), " <- the SMALLEST achievable distance, not zero")
print("A^T (A x - b) =", A.T @ (A @ x - b), " <- ~0: residual is orthogonal to every column")

# np.linalg.solve would refuse: A is not square
print("A.shape =", A.shape, "-> np.linalg.solve cannot be used here")

## 🎯 Problem 3.1 — Least-squares solution

**Given:** an `m × n` matrix `A` with `m > n` and a right-hand side `b` of length `m`.

**Required:** write `least_squares_solution(A, b)` returning the vector `x` (shape `(n,)`) that **minimizes** $\|Ax - b\|$.

**Expected output:** the first element of `np.linalg.lstsq(A, b, rcond=None)`, unpacked so the caller gets just `x`.

In [ ]:
def least_squares_solution(A, b):
    """Least-squares solution of an overdetermined system Ax ~= b.

    Args:
        A: coefficient matrix, shape (m, n) with m > n
        b: right-hand side, shape (m,)

    Returns:
        x: best-fit vector, shape (n,)
    """
    # TODO: Your code here
    pass

# Demo call
demo = generate_linear_system(10, 3, seed=2)
if demo is not None:
    demo_A, demo_b = demo
    demo_x = least_squares_solution(demo_A, demo_b)
    if demo_x is not None:
        print("x =", demo_x, "| ||A x - b|| =", np.linalg.norm(demo_A @ demo_x - demo_b))
    else:
        print("Implement least_squares_solution to power this demo.")
else:
    print("Finish Problem 1.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — which tool</summary>

For m > n no exact solution exists, so you need the *best-fit* solver shown in this Part's worked example — not `np.linalg.solve`.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

`np.linalg.lstsq(A, b, rcond=None)` returns a tuple whose FIRST element is the solution x. Unpack the rest away with `x, *_ = ...` and return `x`.
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def least_squares_solution(A, b):
    """Least-squares solution of an overdetermined system Ax ~= b."""
    x, *_ = np.linalg.lstsq(A, b, rcond=None)
    return x
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.1
res = generate_linear_system(10, 3, seed=2)
assert res is not None, "❌ generate_linear_system returned None — finish Problem 1.1 first."
A_od, b_od = res
x_od = least_squares_solution(A_od, b_od)
assert x_od is not None, "❌ least_squares_solution returned None — replace the 'pass'. See Hint 2 in Part 3."
x_od = np.asarray(x_od, dtype=float)
assert x_od.shape == (3,), f"❌ x has shape {x_od.shape}, expected (3,) for a 10x3 system."
# The signature of an optimal least-squares fit: the residual is orthogonal to
# every column of A (the normal equations). For a random inconsistent system the
# residual itself is NOT ~0 — orthogonality is the property that must be ~0.
assert np.linalg.norm(A_od.T @ (A_od @ x_od - b_od)) < 1e-8, "❌ A^T(Ax - b) should be ~0 for the lstsq optimum — did you return the FIRST output of np.linalg.lstsq? See Hint 2 in Part 3."
r_zero = np.linalg.norm(b_od)  # residual of the naive guess x = 0
assert np.linalg.norm(A_od @ x_od - b_od) <= r_zero + 1e-12, "❌ Your x should fit no worse than the zero vector."
print("✅ Problem 3.1 passed — least squares finds the best-fit x.")

## 🎯 Problem 3.2 — Measuring the residual

**Given:** `A`, `b`, and a candidate solution `x`.

**Required:** write `residual_norm(A, b, x)` returning the 2-norm $\|Ax - b\|$ as a **plain float**. This one number grades any solution: near 0 means $x$ satisfies the system well.

In [ ]:
def residual_norm(A, b, x):
    """2-norm of the residual ||A x - b||.

    Args:
        A: coefficient matrix, shape (m, n)
        b: right-hand side, shape (m,)
        x: candidate solution, shape (n,)

    Returns:
        float: ||A x - b||
    """
    # TODO: Your code here
    pass

# Demo call (uses your Problem 3.1 function)
demo = generate_linear_system(6, 4, seed=3)
if demo is not None:
    demo_A, demo_b = demo
    demo_x = least_squares_solution(demo_A, demo_b)
    if demo_x is not None:
        print("residual of demo solution:", residual_norm(demo_A, demo_b, demo_x))
    else:
        print("Implement Problem 3.1 to power this demo.")
else:
    print("Finish Problem 1.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — what is the residual</summary>

The residual vector is `A @ x - b` — how far your candidate lands from the target. You need a single number summarizing its size.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

`np.linalg.norm(v)` gives the 2-norm (Euclidean length) of a vector, and returns a scalar. Compute the residual vector first, then take its norm — and return it as a float.
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def residual_norm(A, b, x):
    """2-norm of the residual ||A x - b||."""
    return float(np.linalg.norm(A @ x - b))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.2
res = generate_linear_system(6, 4, seed=9)
assert res is not None, "❌ generate_linear_system returned None — finish Problem 1.1 first."
A_r, b_r = res
x_r = least_squares_solution(A_r, b_r)
assert x_r is not None, "❌ least_squares_solution returned None — finish Problem 3.1 first."
res = residual_norm(A_r, b_r, x_r)
assert res is not None, "❌ residual_norm returned None — replace the 'pass'. See Hint 2 in Part 3."
assert np.isscalar(res), "❌ Return a single number (float), not an array — np.linalg.norm already gives a scalar."
assert float(res) >= 0.0, "❌ A norm can never be negative — check your expression A @ x - b."
# For the lstsq optimum the normal equations hold to machine precision:
assert np.linalg.norm(A_r.T @ (A_r @ x_r - b_r)) < 1e-8, "❌ x does not look like the lstsq optimum (A^T(Ax-b) should be ~0) — check Problem 3.1."
bad = residual_norm(A_r, b_r, np.zeros(4))
assert float(bad) > 0.1, "❌ The residual of the zero vector should be clearly nonzero (about ||b||)."
assert abs(float(residual_norm(A_r, b_r, x_r)) - float(np.linalg.norm(A_r @ x_r - b_r))) < 1e-12, "❌ Should equal np.linalg.norm(A @ x - b)."
print("✅ Problem 3.2 passed — you can measure solution quality.")

## Part 4: Residuals, conditioning & a mini-challenge (≈10 min)

The residual $\|Ax - b\|$ is your quality meter: **≈ 0 for square systems**, **minimal but nonzero** for least squares. One caveat: an **ill-conditioned** $A$ can make a computed $x$ very sensitive to tiny round-off changes.

- `np.linalg.cond(A)` measures this sensitivity: near 1 is excellent; $\gtrsim 10^8$ means treat results with suspicion; singular → infinite.
- Random uniform matrices are almost always well conditioned — which is why your square residuals come out at $10^{-16}$.
- **Mini-challenge:** chain your four functions — generate, solve (auto-picking the method), measure — across a list of sizes. This is exactly the loop Lab Assignment 01 asks you to run.

In [ ]:
# --- Worked example: conditioning + the assignment's two cases ---
A_good = np.array([[2.0, 1.0], [1.0, 3.0]])
A_bad = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-8]])   # nearly singular

print("cond(good) =", np.linalg.cond(A_good))        # small number
print("cond(bad)  =", np.linalg.cond(A_bad))         # huge -> ill-conditioned

# The assignment's two cases, one after another:
for m, n, seed in [(3, 3, 11), (10, 3, 12)]:
    rng = np.random.default_rng(seed)
    A = rng.uniform(-1, 1, size=(m, n))
    b = rng.uniform(-1, 3, size=m)
    x = np.linalg.solve(A, b) if m == n else np.linalg.lstsq(A, b, rcond=None)[0]
    print(f"m={m}, n={n}: ||Ax - b|| = {np.linalg.norm(A @ x - b):.3e}")

## 🎯 Problem 4.1 — Mini-challenge: the experiment runner

**Given:** a list `sizes` of `(m, n)` pairs (each with $m \geq n$) and a starting seed `seed_start`.

**Required:** write `run_experiments(sizes, seed_start=0)` that for each pair (index `i`):

1. generates a system with `generate_linear_system(m, n, seed=seed_start + i)` — a **different seed per run**, like the assignment asks,
2. solves it: `solve_square` when `m == n`, `least_squares_solution` when `m > n`,
3. appends `{'m': m, 'n': n, 'x': x, 'residual': r}` to a results list, where `r` comes from `residual_norm`.

**Returns:** the list of dicts, in the same order as `sizes`.

Finishing this problem means you have already done the core loop of **Lab Assignment 01** — with hints. Nothing in the assignment will surprise you.

In [ ]:
def run_experiments(sizes, seed_start=0):
    """Generate, solve, and grade one random system per (m, n) pair.

    Args:
        sizes: list of (m, n) tuples with m >= n
        seed_start: first seed; pair i uses seed_start + i

    Returns:
        list of dicts with keys 'm', 'n', 'x', 'residual'
    """
    # TODO: Your code here
    pass

# Demo call
demo_results = run_experiments([(3, 3), (10, 3)], seed_start=30)
if demo_results is not None:
    for r in demo_results:
        print(f"m={r['m']}, n={r['n']}: residual = {r['residual']:.3e}")
else:
    print("Implement run_experiments to power this demo.")

<details>
<summary>💡 Hint 1 — shape the loop</summary>

You already own every piece: generation (Problem 1.1), solving (Problems 2.1/3.1), grading (Problem 3.2). You need a loop over `sizes`, an index-based seed, and an `if` that picks the solver.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
results = []
for i, (m, n) in enumerate(sizes):
    A, b = generate_linear_system(m, n, seed=seed_start + i)
    x = solve_square(A, b) if m == n else least_squares_solution(A, b)
    results.append({'m': m, 'n': n, 'x': x, 'residual': residual_norm(A, b, x)})
return results
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def run_experiments(sizes, seed_start=0):
    """Generate, solve, and grade one random system per (m, n) pair."""
    results = []
    for i, (m, n) in enumerate(sizes):
        A, b = generate_linear_system(m, n, seed=seed_start + i)
        if m == n:
            x = solve_square(A, b)               # exact solution
        else:
            x = least_squares_solution(A, b)     # least-squares fit
        results.append({"m": m, "n": n, "x": x,
                        "residual": residual_norm(A, b, x)})
    return results
```
</details>

In [ ]:
# 🧪 Self-check for Problem 4.1
results = run_experiments([(3, 3), (10, 3), (5, 2)], seed_start=20)
assert results is not None and len(results) == 3, "❌ run_experiments should return one dict per (m, n) pair. See Hint 2 in Part 4."
for r in results:
    assert set(r.keys()) >= {"m", "n", "x", "residual"}, f"❌ Each record needs keys m, n, x, residual — got {sorted(r.keys())}."
sq = [r for r in results if r["m"] == r["n"]][0]
od = [r for r in results if r["m"] > r["n"]][0]
assert np.isfinite(sq["residual"]) and sq["residual"] < 1e-8, "❌ The square-system residual should be ~0 — did you call solve_square when m == n?"
assert np.isfinite(od["residual"]) and od["residual"] >= 0.0, "❌ The least-squares residual must be a finite, non-negative number."
assert np.asarray(od["x"]).shape == (od["n"],), "❌ x should have shape (n,)."
assert isinstance(od["residual"], float), "❌ residual should be a plain float (build it with residual_norm)."
r1 = run_experiments([(4, 4)], seed_start=20)
r2 = run_experiments([(4, 4)], seed_start=21)
assert not np.allclose(r1[0]["x"], r2[0]["x"]), "❌ Each run should use seed_start + i so different runs differ."
print("✅ Problem 4.1 passed — the full experiment loop works. Lab Assignment 01 awaits!")

## 🎉 You've completed Lab Activity 1

You have mastered:
- Building reproducible random linear systems ($a \sim U(-1,1)$, $b \sim U(-1,3)$) with `np.random.default_rng`
- Solving square systems exactly with `np.linalg.solve`
- Solving overdetermined systems with `np.linalg.lstsq` — and judging optimality via the orthogonality of the residual
- Measuring quality with $\|Ax - b\|$ and recognizing conditioning (`np.linalg.cond`) as the hidden caveat
- Chaining everything into a reusable experiment runner across sizes ($m \geq n$)

**You are now ready for Lab Assignment 01 on the course portal — the assignment asks for the same techniques without hints.**

💡 **Tip:** restart the kernel and run every cell top-to-bottom once more — fluency comes from repetition, and each 🧪 self-check should print ✅ the second time around.